# Ranking wlasnych zdjec okiem Strykera

Narzedzie uzytkowe: wczytuje zapisanego krytyka (checkpoint z Drive), liczy score dla Twoich zdjec i pokazuje je **od najbardziej do najmniej Strykerowskiego** z czytelnymi miniaturkami.

## Jak uzyc
1. Setup (komorki 0-1)
2. Wgraj zdjecia do `/content/moje_zdjecia/`
3. Wybierz checkpoint (komorka 2) — `krytyk_glowa_hygiene` lub `krytyk_lora_hygiene`
4. Policz ranking (3), obejrzyj galerie (4), kubelki (5), rozklad (6)

## Uwaga interpretacyjna
To jest oko **Strykera** (dokument FSA, lata 30.), nie Twoje. Z testu T4: ceni obecnosc ludzi, zbiorowa dokumentacje; moze nie zgadzac sie z Twoim gustem. Traktuj jako *drugie spojrzenie*, nie wyrocznie.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip uninstall -q -y torchao 2>/dev/null
!pip install -q open_clip_torch peft 2>/dev/null


In [ ]:
from pathlib import Path
import numpy as np, json, os
import torch, torch.nn as nn, open_clip
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

DRIVE = Path('/content/drive/MyDrive/fsa_data')
CKPT_DIR = DRIVE/'krytyk_checkpoints'
MY = Path('/content/moje_zdjecia'); MY.mkdir(exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Wgraj zdjecia do: {MY}')
print(f'Dostepne checkpointy:')
for f in sorted(CKPT_DIR.glob('*_meta.json')):
    m = json.load(open(f))
    print(f"  - {m['run_name']}: {m['backbone']}, LoRA={m['use_lora']}, test_acc={m.get('best_test_acc',0):.3f}")


## 1. Definicja glowy + funkcje wczytania (musi zgadzac sie z treningiem)

In [ ]:
class ScoreHead(nn.Module):
    def __init__(self, dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim,hidden), nn.GELU(), nn.Dropout(0.3), nn.Linear(hidden,1))
    def forward(self,x): return self.net(x).squeeze(-1)

def load_critic(run_name):
    meta = json.load(open(CKPT_DIR/f'{run_name}_meta.json'))
    m, _, prep = open_clip.create_model_and_transforms(meta['backbone'], pretrained=meta['pretrained'])
    m = m.to(device)
    if meta['use_lora']:
        from peft import PeftModel
        m = PeftModel.from_pretrained(m, str(CKPT_DIR/f'{run_name}_lora')).to(device)
    m.eval()
    h = ScoreHead(meta['emb_dim'], meta.get('head_hidden',256)).to(device)
    h.load_state_dict(torch.load(CKPT_DIR/f'{run_name}_head.pt', map_location=device)); h.eval()
    return m, h, prep, meta

@torch.no_grad()
def score_folder(folder, m, h, prep):
    exts=('.jpg','.jpeg','.png','.webp')
    files=sorted([f for f in Path(folder).iterdir() if f.suffix.lower() in exts])
    assert files, f'Brak zdjec w {folder}'
    scores=[]
    for i in range(0,len(files),32):
        batch=files[i:i+32]
        imgs=torch.stack([prep(ImageOps.exif_transpose(Image.open(f)).convert('RGB')) for f in batch]).to(device)
        e=m.encode_image(imgs); e=e/e.norm(dim=-1,keepdim=True)
        scores.append(h(e).cpu().numpy())
    scores=np.concatenate(scores)
    order=np.argsort(scores)[::-1]
    return [files[i] for i in order], scores[order]
print('Funkcje gotowe')


## 2. Wybierz checkpoint

In [ ]:
# Zmien na 'krytyk_lora_hygiene' jesli chcesz wersje LoRA
RUN_NAME = 'krytyk_glowa_hygiene'

model, head, preprocess, meta = load_critic(RUN_NAME)
print(f'Wczytano: {RUN_NAME}')
print(f"  backbone={meta['backbone']}, LoRA={meta['use_lora']}, higiena={meta['hygiene']}")
print(f"  test_acc (na FSA)={meta.get('best_test_acc',0):.3f}")


## 3. Policz ranking Twoich zdjec

In [ ]:
ranked_files, ranked_scores = score_folder(MY, model, head, preprocess)
N = len(ranked_files)
print(f'Zdjec: {N}')
print(f'Score: min={ranked_scores.min():+.3f}, max={ranked_scores.max():+.3f}, '
      f'rozstep={ranked_scores.max()-ranked_scores.min():.3f}')
print(f'\nTop 5:')
for i in range(min(5,N)): print(f'  {i+1}. {ranked_scores[i]:+.3f}  {ranked_files[i].name}')
print(f'Dol 3:')
for i in range(max(0,N-3),N): print(f'  {i+1}. {ranked_scores[i]:+.3f}  {ranked_files[i].name}')


## 4. Galeria z miniaturkami (czytelne kafelki, stronicowana)

In [ ]:
# Ustaw ile pokazac i od ktorej pozycji (paginacja dla duzych zbiorow)
POKAZ = 12        # ile kafelkow naraz
OD = 0            # pozycja startowa (0 = od najlepszych; zmien na N-POKAZ dla najgorszych)
COLS = 4

def galeria(od, ile, tytul):
    sel = list(range(od, min(od+ile, N)))
    rows = (len(sel)+COLS-1)//COLS
    fig, axes = plt.subplots(rows, COLS, figsize=(4.2*COLS, 4.2*rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis('off')
    for k, idx in enumerate(sel):
        pil = ImageOps.exif_transpose(Image.open(ranked_files[idx])).convert('RGB')
        pil.thumbnail((500,500))
        axes[k].imshow(pil)
        # kolor ramki wg pozycji: gora zielona, dol czerwona
        frac = idx/max(N-1,1)
        col = plt.cm.RdYlGn(1-frac)
        for s in axes[k].spines.values(): s.set_visible(True); s.set_edgecolor(col); s.set_linewidth(4)
        axes[k].set_title(f'#{idx+1}/{N}   score={ranked_scores[idx]:+.3f}', fontsize=11, color='black')
    plt.suptitle(tytul, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

galeria(OD, POKAZ, f'Ranking Strykera — pozycje {OD+1}-{min(OD+POKAZ,N)} (zielone=lepsze)')


### 4b. Pokaz najgorsze (dol rankingu)

In [ ]:
galeria(max(0,N-POKAZ), POKAZ, f'Dol rankingu — najmniej Strykerowskie (pozycje {max(1,N-POKAZ+1)}-{N})')


## 5. Kubelki — gora / srodek / dol wg percentyli

In [ ]:
# Podzial na trzy grupy: top 25%, srodek, dol 25%
q_hi, q_lo = np.percentile(ranked_scores, [75, 25])
gora = [f for f,s in zip(ranked_files,ranked_scores) if s>=q_hi]
srodek = [f for f,s in zip(ranked_files,ranked_scores) if q_lo<s<q_hi]
dol = [f for f,s in zip(ranked_files,ranked_scores) if s<=q_lo]
print(f'GORA (top 25%, score>={q_hi:+.3f}): {len(gora)} zdjec  -> kandydaci do zachowania')
print(f'SRODEK: {len(srodek)} zdjec')
print(f'DOL (dol 25%, score<={q_lo:+.3f}): {len(dol)} zdjec  -> kandydaci do odrzucenia')
print()
print('Nazwy z GORY:')
for f in gora[:15]: print(f'  {f.name}')


## 6. Rozklad score — czy os w ogole rozroznia Twoje zdjecia

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].hist(ranked_scores, bins=min(30,N//2), color='steelblue', edgecolor='white')
ax[0].axvline(q_hi, color='green', ls='--', label=f'top 25% ({q_hi:+.2f})')
ax[0].axvline(q_lo, color='red', ls='--', label=f'dol 25% ({q_lo:+.2f})')
ax[0].set_xlabel('score Strykera'); ax[0].set_ylabel('liczba zdjec'); ax[0].legend()
ax[0].set_title('Rozklad score')
ax[1].plot(range(1,N+1), ranked_scores, color='navy')
ax[1].set_xlabel('pozycja w rankingu'); ax[1].set_ylabel('score')
ax[1].set_title('Krzywa rankingu')
plt.tight_layout(); plt.show()

rozstep = ranked_scores.max()-ranked_scores.min()
std = ranked_scores.std()
print(f'Rozstep score: {rozstep:.3f}, odchylenie: {std:.3f}')
if std < 0.1:
    print('-> UWAGA: maly rozrzut. Os Strykera slabo rozroznia Twoje zdjecia')
    print('   (mozliwe: Twoj gatunek/styl daleki od dokumentu FSA — patrz T4).')
else:
    print('-> Os wyraznie rozroznia Twoje zdjecia — ranking jest informatywny.')


## 7. Zapisz ranking do CSV (opcjonalnie)

In [ ]:
import csv
out_csv = MY/'ranking_strykera.csv'
with open(out_csv,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['pozycja','score','plik'])
    for i,(fn,sc) in enumerate(zip(ranked_files,ranked_scores)):
        w.writerow([i+1, f'{sc:.4f}', fn.name])
print(f'Zapisano ranking: {out_csv}')
print('Mozesz go pobrac z panelu plikow Colaba.')
